In [1]:
import numpy as np

data = np.load("processed/rl_train.npz")

rewards = data["rewards"]

reward_rate = rewards.mean()
print(f"Reward rate: {reward_rate:.4f}")
print(f"Reward %: {reward_rate * 100:.2f}%")

Reward rate: 0.8088
Reward %: 80.88%


In [2]:
terminals = data["terminals"]

episode_rewards = []
current_sum = 0
current_len = 0

for r, done in zip(rewards, terminals):
    current_sum += r
    current_len += 1
    if done:
        episode_rewards.append(current_sum / current_len)
        current_sum = 0
        current_len = 0

print(f"Mean reward per episode: {np.mean(episode_rewards):.4f}")

Mean reward per episode: 0.8045


In [3]:
H = 5
future_returns = []

for i in range(len(rewards)):
    total = 0
    for k in range(1, H+1):
        if i+k >= len(rewards):
            break
        total += rewards[i+k]
    future_returns.append(total)

print("Mean future 5-step return:", np.mean(future_returns))

Mean future 5-step return: 4.043746175248044


In [5]:
import numpy as np
from collections import defaultdict

d = np.load("processed/rl_train.npz")
a = d["actions"].flatten()
r = d["rewards"]

sum_r = defaultdict(float)
cnt = defaultdict(int)

for ai, ri in zip(a, r):
    sum_r[int(ai)] += float(ri)
    cnt[int(ai)] += 1

rates = np.array([sum_r[k] / cnt[k] for k in cnt.keys() if cnt[k] >= 50])  # filtra acciones con soporte
print("Num actions (cnt>=50):", len(rates))
print("Mean rate:", rates.mean())
print("Std rate:", rates.std())
print("P10/P50/P90:", np.quantile(rates, [0.1, 0.5, 0.9]))

Num actions (cnt>=50): 4279
Mean rate: 0.8129859498954127
Std rate: 0.09342825366386018
P10/P50/P90: [0.68301394 0.82894737 0.91948413]


In [6]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from collections import defaultdict

PATH = "processed/rl_train.npz"
K = 200                 # nº clusters de estados
MIN_CNT = 20            # mínimo soporte (acción, cluster)
MAX_ITEMS_ANALYZE = 5000  # analiza solo top acciones para ir rápido (opcional)

d = np.load(PATH)
S = d["observations"].astype(np.float32)
A = d["actions"].flatten().astype(np.int32)
R = d["rewards"].astype(np.float32)

# 1) Clusteriza estados
kmeans = MiniBatchKMeans(n_clusters=K, batch_size=16384, random_state=0)
C = kmeans.fit_predict(S)  # cluster id por transición

# 2) Top acciones más frecuentes (para reducir coste)
counts_a = np.bincount(A)
top_actions = np.argsort(counts_a)[::-1][:MAX_ITEMS_ANALYZE]
top_set = set(top_actions.tolist())

# 3) Global rate por acción (solo top)
sum_r_a = defaultdict(float)
cnt_a = defaultdict(int)
for a, r in zip(A, R):
    if a in top_set:
        sum_r_a[a] += float(r)
        cnt_a[a] += 1
rate_global = {a: sum_r_a[a]/cnt_a[a] for a in cnt_a.keys()}

# 4) Rate por (cluster, acción)
sum_rca = defaultdict(float)
cnt_rca = defaultdict(int)

for c, a, r in zip(C, A, R):
    if a in top_set:
        key = (int(c), int(a))
        sum_rca[key] += float(r)
        cnt_rca[key] += 1

# 5) Mide "dependencia de estado": desviación media |rate(c,a)-rate(a)|
diffs = []
for (c, a), cnt in cnt_rca.items():
    if cnt >= MIN_CNT:
        diffs.append(abs(sum_rca[(c,a)]/cnt - rate_global[a]))

diffs = np.array(diffs)
print("Pairs (cluster,action) with support:", len(diffs))
print("Mean |rate_cluster - rate_global|:", diffs.mean())
print("P50/P90/P99:", np.quantile(diffs, [0.5, 0.9, 0.99]))

Pairs (cluster,action) with support: 971
Mean |rate_cluster - rate_global|: 0.03517527807131021
P50/P90/P99: [0.02648912 0.07592872 0.1411041 ]


In [7]:
import numpy as np
from collections import defaultdict

# Reutiliza C, A, R, K, top_set, MIN_CNT del script anterior

# Acumula rates por cluster para acciones con soporte
cluster_action_sum = [defaultdict(float) for _ in range(K)]
cluster_action_cnt = [defaultdict(int) for _ in range(K)]

for c, a, r in zip(C, A, R):
    if a in top_set:
        cluster_action_sum[c][a] += float(r)
        cluster_action_cnt[c][a] += 1

top10_per_cluster = []
for c in range(K):
    rates = []
    for a, cnt in cluster_action_cnt[c].items():
        if cnt >= MIN_CNT:
            rates.append((cluster_action_sum[c][a]/cnt, a, cnt))
    rates.sort(reverse=True)
    top10 = [a for _, a, _ in rates[:10]]
    top10_per_cluster.append(set(top10))

# Similaridad Jaccard promedio entre tops de clusters
def jaccard(s1, s2):
    if not s1 and not s2: return 1.0
    return len(s1 & s2) / max(1, len(s1 | s2))

sims = []
for i in range(K):
    for j in range(i+1, K):
        sims.append(jaccard(top10_per_cluster[i], top10_per_cluster[j]))
sims = np.array(sims)

print("Avg Jaccard(top10) across clusters:", sims.mean())
print("P10/P50/P90:", np.quantile(sims, [0.1, 0.5, 0.9]))

Avg Jaccard(top10) across clusters: 0.07226524787717362
P10/P50/P90: [0.   0.   0.25]


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 3
BATCH = 8192
LR = 1e-3
EMB = 64

d = np.load("processed/rl_train.npz")
S = d["observations"].astype(np.float32)
A = d["actions"].flatten().astype(np.int64)
R = d["rewards"].astype(np.float32)

# split rápido
idx = np.arange(len(R))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=0)

S_tr = torch.tensor(S[train_idx])
A_tr = torch.tensor(A[train_idx])
R_tr = torch.tensor(R[train_idx]).unsqueeze(1)

S_te = torch.tensor(S[test_idx])
A_te = torch.tensor(A[test_idx])
R_te = R[test_idx]

num_items = int(A.max()) + 1
state_dim = S.shape[1]

class ActionOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(num_items + 1, EMB)
        self.mlp = nn.Sequential(nn.Linear(EMB, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, a):
        x = self.emb(a)
        return self.mlp(x)

class StateAction(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(num_items + 1, EMB)
        self.mlp = nn.Sequential(
            nn.Linear(state_dim + EMB, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, s, a):
        x = torch.cat([s, self.emb(a)], dim=1)
        return self.mlp(x)

def train(model, mode="sa"):
    model = model.to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()

    if mode == "a":
        ds = torch.utils.data.TensorDataset(A_tr, R_tr)
    else:
        ds = torch.utils.data.TensorDataset(S_tr, A_tr, R_tr)

    dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True)

    model.train()
    for _ in range(EPOCHS):
        for batch in dl:
            opt.zero_grad()
            if mode == "a":
                a, y = [t.to(DEVICE) for t in batch]
                logits = model(a)
            else:
                s, a, y = [t.to(DEVICE) for t in batch]
                logits = model(s, a)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()
    return model

# Entrena
mA = train(ActionOnly(), mode="a")
mB = train(StateAction(), mode="sa")

# Evalúa AUC
mA.eval(); mB.eval()
with torch.no_grad():
    pA = torch.sigmoid(mA(A_te.to(DEVICE))).cpu().numpy().ravel()
    pB = torch.sigmoid(mB(S_te.to(DEVICE), A_te.to(DEVICE))).cpu().numpy().ravel()

R_te_bin = (R_te > 0.5).astype(np.int32)

aucA = roc_auc_score(R_te_bin, pA)
aucB = roc_auc_score(R_te_bin, pB)
print("AUC action-only:", aucA)
print("AUC state+action:", aucB)
print("ΔAUC:", aucB - aucA)

ValueError: continuous format is not supported

In [1]:
import pandas as pd
import numpy as np
import json
from collections import Counter

# ============================================================
# 1️⃣ CARGA PARA .json (JSON Lines)
# ============================================================

def load_json(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Ajusta la ruta aquí
df = load_json("raw/Electronics_5.json")

print("Shape raw:", df.shape)
print("Columns:", df.columns.tolist())
print("\n")

# ============================================================
# 2️⃣ LIMPIEZA BÁSICA
# ============================================================

df = df.dropna(subset=["reviewerID", "asin", "unixReviewTime", "overall"]).copy()

df["overall"] = df["overall"].astype(float)
df["overall"] = df["overall"].clip(1, 5)

df["helpful_yes"] = df["helpful"].apply(lambda x: x[0] if isinstance(x, list) else 0)
df["helpful_total"] = df["helpful"].apply(lambda x: x[1] if isinstance(x, list) else 0)
df["helpful_ratio"] = np.where(
    df["helpful_total"] > 0,
    df["helpful_yes"] / df["helpful_total"],
    0.0
)

print("After cleaning:", df.shape)
print("\n")

# ============================================================
# 3️⃣ ESTADÍSTICAS GENERALES
# ============================================================

n_users = df["reviewerID"].nunique()
n_items = df["asin"].nunique()
n_interactions = len(df)

print("Num users:", n_users)
print("Num items:", n_items)
print("Num interactions:", n_interactions)
print("\n")

# ============================================================
# 4️⃣ DISTRIBUCIÓN DE RATINGS
# ============================================================

print("Rating distribution:")
print(df["overall"].value_counts(normalize=True).sort_index())
print("\n")

print("Rating mean:", df["overall"].mean())
print("Rating std:", df["overall"].std())
print("\n")

# ============================================================
# 5️⃣ LONGITUDES DE HISTORIAL POR USUARIO
# ============================================================

user_lengths = df.groupby("reviewerID").size()

print("User interaction length stats:")
print(user_lengths.describe(percentiles=[.5, .75, .9, .95, .99]))
print("\n")

print("Users with <3 interactions:", (user_lengths < 3).mean())
print("Users with <5 interactions:", (user_lengths < 5).mean())
print("\n")

# ============================================================
# 6️⃣ LONGITUDES POR ITEM
# ============================================================

item_lengths = df.groupby("asin").size()

print("Item frequency stats:")
print(item_lengths.describe(percentiles=[.5, .75, .9, .95, .99]))
print("\n")

print("Items with <5 interactions:", (item_lengths < 5).mean())
print("Items with <10 interactions:", (item_lengths < 10).mean())
print("\n")

# ============================================================
# 7️⃣ ANÁLISIS DE GAPS TEMPORALES (para sesiones)
# ============================================================

df_sorted = df.sort_values(["reviewerID", "unixReviewTime"]).copy()

df_sorted["prev_time"] = df_sorted.groupby("reviewerID")["unixReviewTime"].shift(1)
df_sorted["time_gap_days"] = (
    (df_sorted["unixReviewTime"] - df_sorted["prev_time"]) / (60 * 60 * 24)
)

gaps = df_sorted["time_gap_days"].dropna()

print("Time gap stats (days):")
print(gaps.describe(percentiles=[.5, .75, .9, .95, .99]))
print("\n")

print("Percent gaps > 7 days:", (gaps > 7).mean())
print("Percent gaps > 30 days:", (gaps > 30).mean())
print("Percent gaps > 90 days:", (gaps > 90).mean())
print("\n")

# ============================================================
# 8️⃣ DUPLICADOS
# ============================================================

dup_user_item = df.duplicated(subset=["reviewerID", "asin"]).mean()
dup_user_item_time = df.duplicated(subset=["reviewerID", "asin", "unixReviewTime"]).mean()

print("Duplicate (user,item):", dup_user_item)
print("Duplicate (user,item,time):", dup_user_item_time)
print("\n")

# ============================================================
# 9️⃣ HELPFUL SIGNAL
# ============================================================

print("Helpful total stats:")
print(df["helpful_total"].describe(percentiles=[.5, .75, .9, .95, .99]))
print("\n")

print("Percent helpful_total = 0:", (df["helpful_total"] == 0).mean())
print("Helpful ratio mean:", df["helpful_ratio"].mean())
print("\n")

# ============================================================
# 🔟 SPARSITY
# ============================================================

density = n_interactions / (n_users * n_items)
print("Matrix density:", density)
print("\n")

print("DONE.")

Shape raw: (1689188, 9)
Columns: ['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']


After cleaning: (1689188, 12)


Num users: 192403
Num items: 63001
Num interactions: 1689188


Rating distribution:
overall
1.0    0.064365
2.0    0.048626
3.0    0.084216
4.0    0.205448
5.0    0.597344
Name: proportion, dtype: float64


Rating mean: 4.222779228836577
Rating std: 1.185631797800484


User interaction length stats:
count    192403.000000
mean          8.779427
std           8.263942
min           5.000000
50%           7.000000
75%           9.000000
90%          14.000000
95%          19.000000
99%          39.000000
max         431.000000
dtype: float64


Users with <3 interactions: 0.0
Users with <5 interactions: 0.0


Item frequency stats:
count    63001.000000
mean        26.812082
std         75.821107
min          5.000000
50%         11.000000
75%         22.000000
90%         51.000000
95%         88.000000
99%

In [2]:
import pandas as pd
import numpy as np

PAD_ITEM_ID = 0

# ============================================================
# 1️⃣ Mapear IDs (PAD=0 reservado)
# ============================================================

user2id = {u:i for i,u in enumerate(df["reviewerID"].unique())}
asin2id = {a:i+1 for i,a in enumerate(df["asin"].unique())}  # +1 para reservar 0 como PAD

events = df[["reviewerID","asin","unixReviewTime","overall",
             "helpful_ratio","helpful_total"]].copy()

events["user_id"] = events["reviewerID"].map(user2id).astype(np.int32)
events["item_id"] = events["asin"].map(asin2id).astype(np.int32)

# ============================================================
# 2️⃣ Reward denso
# ============================================================

events["reward"] = (
    (events["overall"].astype(np.float32) - 1.0) / 4.0
).clip(0.0, 1.0)

# ============================================================
# 3️⃣ Orden temporal (CRÍTICO)
# ============================================================

events = events.sort_values(
    ["user_id", "unixReviewTime"],
    kind="mergesort"
).reset_index(drop=True)

# ============================================================
# 4️⃣ Episodio = usuario completo
# ============================================================

events["session_id"] = events["user_id"]

events["pos_in_session"] = (
    events.groupby("user_id").cumcount().astype(np.int32)
)

events["session_len"] = (
    events.groupby("user_id")["item_id"]
    .transform("size")
    .astype(np.int32)
)

# ============================================================
# 5️⃣ Stats
# ============================================================

num_sessions = events["session_id"].nunique()
sess_lens = events.groupby("session_id").size()

print("Events:", len(events))
print("Users:", events["user_id"].nunique())
print("Items:", events["item_id"].nunique())
print("Episodes (users):", num_sessions)

print("\nEpisode length stats:")
print(sess_lens.describe(percentiles=[.5,.75,.9,.95,.99]))

print("\n% episodes len=1:", (sess_lens==1).mean())
print("% episodes len<3:", (sess_lens<3).mean())

# ============================================================
# 6️⃣ Time-based split por usuario
# ============================================================

events["rank_in_user"] = events.groupby("user_id").cumcount()
events["user_len"] = events.groupby("user_id")["item_id"].transform("size")

test_cut = (events["user_len"] * 0.9).astype(int)
val_cut  = (events["user_len"] * 0.8).astype(int)

events["split"] = "train"
events.loc[events["rank_in_user"] >= test_cut, "split"] = "test"
events.loc[
    (events["rank_in_user"] >= val_cut) &
    (events["rank_in_user"] < test_cut),
    "split"
] = "val"

print("\nSplit ratios:")
print(events["split"].value_counts(normalize=True))

Events: 1689188
Users: 192403
Items: 63001
Episodes (users): 192403

Episode length stats:
count    192403.000000
mean          8.779427
std           8.263942
min           5.000000
50%           7.000000
75%           9.000000
90%          14.000000
95%          19.000000
99%          39.000000
max         431.000000
dtype: float64

% episodes len=1: 0.0
% episodes len<3: 0.0

Split ratios:
split
train    0.758794
test     0.145851
val      0.095355
Name: proportion, dtype: float64


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 50

class NextItemDataset(Dataset):
    def __init__(self, events_df, max_len=50):
        self.max_len = max_len
        self.groups = []
        # usaremos sesiones para que el modelo aprenda contexto local
        for sid, g in events_df.groupby("session_id", sort=False):
            items = g["item_id"].to_numpy(dtype=np.int64)
            if len(items) < 2:
                continue
            self.groups.append(items)

        # índice a (grupo, t)
        self.index = []
        for gi, items in enumerate(self.groups):
            for t in range(1, len(items)):  # predecir items[t] usando history <t
                self.index.append((gi, t))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        gi, t = self.index[idx]
        items = self.groups[gi]
        hist = items[:t]
        target = items[t]

        # recorte a max_len
        hist = hist[-self.max_len:]
        x = np.full((self.max_len,), PAD_ITEM_ID, dtype=np.int64)
        x[-len(hist):] = hist
        return torch.from_numpy(x), torch.tensor(target, dtype=torch.long)

train_df = events[events["split"]=="train"]
train_ds = NextItemDataset(train_df, max_len=MAX_SEQ_LEN)
print("Train next-item examples:", len(train_ds))

loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0, pin_memory=False)
xb, yb = next(iter(loader))
print("Batch x:", xb.shape, "Batch y:", yb.shape)

Train next-item examples: 1089342
Batch x: torch.Size([512, 50]) Batch y: torch.Size([512])


In [4]:
import torch
import torch.nn as nn
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_ITEMS = events["item_id"].nunique() + 1  # +1 por si acaso (incluye PAD=0)

EMB_DIM = 64
HID_DIM = 128

class GRUEncoder(nn.Module):
    def __init__(self, num_items, emb_dim=64, hid_dim=128, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(num_items, emb_dim, padding_idx=pad_id)
        self.gru = nn.GRU(input_size=emb_dim, hidden_size=hid_dim, batch_first=True)
    def forward(self, x, h0=None):
        # x: [B, T]
        e = self.emb(x)
        out, h = self.gru(e, h0)  # h: [1, B, H]
        return out, h

class NextItemHead(nn.Module):
    def __init__(self, hid_dim, num_items):
        super().__init__()
        self.fc = nn.Linear(hid_dim, num_items)
    def forward(self, h_last):
        return self.fc(h_last)

encoder = GRUEncoder(NUM_ITEMS, EMB_DIM, HID_DIM, pad_id=PAD_ITEM_ID).to(device)
head = NextItemHead(HID_DIM, NUM_ITEMS).to(device)

# --- entrenamiento opcional (mínimo) ---
def train_next_item(epochs=2, lr=1e-3):
    encoder.train(); head.train()
    opt = torch.optim.Adam(list(encoder.parameters()) + list(head.parameters()), lr=lr)
    ce = nn.CrossEntropyLoss()

    for ep in range(1, epochs+1):
        total_loss = 0.0
        n = 0
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            _, h = encoder(xb)            # h: [1,B,H]
            h_last = h[0]                 # [B,H]
            logits = head(h_last)         # [B,num_items]
            loss = ce(logits, yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * xb.size(0)
            n += xb.size(0)
        print(f"epoch {ep} loss {total_loss/n:.4f}")

# Descomenta para entrenar algo
train_next_item(epochs=3, lr=1e-3)

# --- generar dataset RL ---
@torch.no_grad()
def build_offline_rl_dataset(events_df):
    encoder.eval()

    obs_list = []
    next_obs_list = []
    act_list = []
    rew_list = []
    done_list = []

    for uid, g in events_df.groupby("session_id", sort=False):
        items = g["item_id"].to_numpy(dtype=np.int64)
        rewards = g["reward"].to_numpy(dtype=np.float32)
        T = len(items)

        if T < 2:
            continue

        # inicializa hidden
        h = torch.zeros((1, 1, HID_DIM), device=device)

        # primero consumimos item 0 para tener estado personalizado
        x0 = torch.tensor([[items[0]]], dtype=torch.long, device=device)
        _, h = encoder(x0, h)

        for t in range(1, T):

            s = h[0,0].detach().cpu().numpy().astype(np.float32)
            a = items[t]
            r = rewards[t]

            # actualizar
            xt = torch.tensor([[a]], dtype=torch.long, device=device)
            _, h2 = encoder(xt, h)

            s2 = h2[0,0].detach().cpu().numpy().astype(np.float32)

            done = 1.0 if (t == T-1) else 0.0

            obs_list.append(s)
            act_list.append(a)
            rew_list.append(r)
            next_obs_list.append(s2)
            done_list.append(done)

            h = h2

    dataset = {
        "observations": np.stack(obs_list, axis=0),
        "actions": np.array(act_list, dtype=np.int32),
        "rewards": np.array(rew_list, dtype=np.float32),
        "next_observations": np.stack(next_obs_list, axis=0),
        "terminals": np.array(done_list, dtype=np.float32),
        "num_items": np.int64(NUM_ITEMS),
    }
    return dataset
# Construye RL dataset SOLO en train (recomendado)
rl_train = build_offline_rl_dataset(events[events["split"]=="train"])

print("RL keys:", rl_train.keys())
print("observations:", rl_train["observations"].shape, rl_train["observations"].dtype)
print("actions:", rl_train["actions"].shape, rl_train["actions"].dtype, rl_train["actions"].min(), rl_train["actions"].max())
print("rewards:", rl_train["rewards"].shape, rl_train["rewards"].dtype, rl_train["rewards"].min(), rl_train["rewards"].max(), rl_train["rewards"].mean())
print("terminals:", rl_train["terminals"].shape, rl_train["terminals"].sum(), "avg ep len:", len(rl_train["terminals"]) / rl_train["terminals"].sum())

epoch 1 loss 10.2045
epoch 2 loss 9.6583
epoch 3 loss 9.0666
RL keys: dict_keys(['observations', 'actions', 'rewards', 'next_observations', 'terminals', 'num_items'])
observations: (1089342, 128) float32
actions: (1089342,) int32 1 62997
rewards: (1089342,) float32 0.0 1.0 0.8089879
terminals: (1089342,) 192403.0 avg ep len: 5.6617723


In [5]:
obs = rl_train["observations"]
mean = obs.mean(axis=0, keepdims=True)
std = obs.std(axis=0, keepdims=True) + 1e-6

rl_train["observations"] = (rl_train["observations"] - mean) / std
rl_train["next_observations"] = (rl_train["next_observations"] - mean) / std

In [6]:
np.savez_compressed(
    "electronics_rl_train.npz",
    observations=rl_train["observations"],
    actions=rl_train["actions"],
    rewards=rl_train["rewards"],
    next_observations=rl_train["next_observations"],
    terminals=rl_train["terminals"],
    num_items=rl_train["num_items"],
)

In [ ]:
import numpy as np

data = np.load("processed/offline_rl_candidates.npz")

print("Reward mean:", data["rewards"].mean())
print("Terminals mean:", data["terminals"].mean())
print("Unique rewards:", np.unique(data["rewards"]))

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_json("raw/Electronics_5.json", lines=True)

df = df.sort_values(["reviewerID", "unixReviewTime"])

print(df.head())
print("\nSorted OK")

df = df.sort_values(["reviewerID", "unixReviewTime"])

user2id = {u:i for i,u in enumerate(df["reviewerID"].unique())}
item2id = {a:i for i,a in enumerate(df["asin"].unique())}

df["user_id"] = df["reviewerID"].map(user2id)
df["item_id"] = df["asin"].map(item2id)

print("Users:", len(user2id))
print("Items:", len(item2id))

np.savez("maps_ids.npz", user2id=user2id, item2id=item2id)

print("Mapping saved.")

df = df.sort_values(["reviewerID", "unixReviewTime"])

user2id = {u:i for i,u in enumerate(df["reviewerID"].unique())}
item2id = {a:i for i,a in enumerate(df["asin"].unique())}

df["user_id"] = df["reviewerID"].map(user2id)
df["item_id"] = df["asin"].map(item2id)

sequences = df.groupby("user_id")["item_id"].apply(list)

print("Example sequence:", sequences.iloc[0][:10])
print("Num sequences:", len(sequences))

In [ ]:
import pandas as pd

df = pd.read_json("raw/Electronics_5.json", lines=True)

print("Total rows:", len(df))
print("Unique users:", df["reviewerID"].nunique())
print("Unique items:", df["asin"].nunique())

print("\nReviews per user stats:")
print(df.groupby("reviewerID").size().describe())

In [ ]:
import numpy as np
import json
from collections import Counter
from pprint import pprint

RAW_PATH = "raw/Electronics_5.json"
N_EXAMPLES = 3

def main():
    print("# Leyendo Electronics_5 RAW...\n")

    key_counter = Counter()
    examples = []

    with open(RAW_PATH, "r") as f:
        for i, line in enumerate(f):
            review = json.loads(line)

            if i < N_EXAMPLES:
                examples.append(review)

            key_counter.update(review.keys())

            if i >= 10000:  # solo escanear primeras 10k líneas para ver claves
                break

    print("=" * 80)
    print("## EJEMPLOS COMPLETOS")
    print("=" * 80)

    for i, ex in enumerate(examples):
        print(f"\n--- Ejemplo {i+1} ---")
        pprint(ex)

    print("\n" + "=" * 80)
    print("## CLAVES ENCONTRADAS")
    print("=" * 80)

    for key, count in key_counter.items():
        print(f"{key:20} -> presente en {count} ejemplos")

    print("\n" + "=" * 80)
    print("## TIPOS DE DATOS DEL PRIMER EJEMPLO")
    print("=" * 80)

    first = examples[0]
    for key, value in first.items():
        print(f"{key:20} -> {type(value)}")

main()

print("=" * 80)
print("# Procesando MDP....")
print("=" * 80)

print("=" * 80)
print("## Keys")
print("=" * 80)
data = np.load("processed/offline_rl_dataset.npz")

print("Keys:", data.files)
print()

for k in data.files:
    arr = data[k]
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")
    print("  min:", arr.min() if arr.ndim==1 else "...")
    print("  max:", arr.max() if arr.ndim==1 else "...")
    print()

print("=" * 80)
print("## Terminals")
print("=" * 80)

terminals = data["terminals"]
print("Num transitions:", len(terminals))
print("Num terminals:", terminals.sum())
print("Avg episode length:", len(terminals) / terminals.sum())

print("=" * 80)
print("## Rewards")
print("=" * 80)

r = data["rewards"]

print("Reward mean:", r.mean())
print("Reward std:", r.std())
print("Reward > 0 ratio:", (r > 0).mean())
print("Unique rewards:", np.unique(r)[:20])

print("=" * 80)
print("## Actions")
print("=" * 80)

actions = data["actions"]

print("Num unique actions in dataset:", len(np.unique(actions)))
print("Min action:", actions.min())
print("Max action:", actions.max())

print("=" * 80)
print("## Observations")
print("=" * 80)

obs = data["observations"]

print("State dim:", obs.shape[1])
print("Example state norm mean:", np.linalg.norm(obs, axis=1).mean())

import pandas as pd

print("=" * 80)
print("## events_clean")
print("=" * 80)

df = pd.read_parquet("processed/events_clean.parquet")
print(df.head())
print(df.columns)
print("Num rows:", len(df))

print("=" * 80)
print("## embedding_matrix")
print("=" * 80)

emb = np.load("processed/item_embedding_matrix.npy")
print("Embedding matrix shape:", emb.shape)

In [ ]:

import json
from collections import Counter
from pprint import pprint

RAW_PATH = "raw/Electronics_5.json"
N_EXAMPLES = 3

def main():
    print("Leyendo ejemplos...\n")

    key_counter = Counter()
    examples = []

    with open(RAW_PATH, "r") as f:
        for i, line in enumerate(f):
            review = json.loads(line)

            if i < N_EXAMPLES:
                examples.append(review)

            key_counter.update(review.keys())

            if i >= 10000:  # solo escanear primeras 10k líneas para ver claves
                break

    print("=" * 80)
    print("📌 EJEMPLOS COMPLETOS")
    print("=" * 80)

    for i, ex in enumerate(examples):
        print(f"\n--- Ejemplo {i+1} ---")
        pprint(ex)

    print("\n" + "=" * 80)
    print("📌 CLAVES ENCONTRADAS")
    print("=" * 80)

    for key, count in key_counter.items():
        print(f"{key:20} -> presente en {count} ejemplos")

    print("\n" + "=" * 80)
    print("📌 TIPOS DE DATOS DEL PRIMER EJEMPLO")
    print("=" * 80)

    first = examples[0]
    for key, value in first.items():
        print(f"{key:20} -> {type(value)}")


if __name__ == "__main__":
    main()

In [ ]:
import json
from collections import defaultdict, Counter
from tqdm import tqdm

RAW_PATH = "raw/Electronics_5.json"

def main():
    item_review_count = defaultdict(int)
    item_has_text = defaultdict(bool)
    item_has_summary = defaultdict(bool)
    item_empty_text_count = defaultdict(int)

    total_reviews = 0

    print("Analizando dataset...")

    with open(RAW_PATH, "r") as f:
        for line in tqdm(f):
            review = json.loads(line)

            asin = review["asin"]
            review_text = review.get("reviewText", "")
            summary = review.get("summary", "")

            total_reviews += 1
            item_review_count[asin] += 1

            # Verificar reviewText
            if isinstance(review_text, str) and review_text.strip() != "":
                item_has_text[asin] = True
            else:
                item_empty_text_count[asin] += 1

            # Verificar summary
            if isinstance(summary, str) and summary.strip() != "":
                item_has_summary[asin] = True

    total_items = len(item_review_count)

    items_without_text = [
        asin for asin in item_review_count
        if not item_has_text[asin]
    ]

    items_without_summary = [
        asin for asin in item_review_count
        if not item_has_summary[asin]
    ]

    print("\n" + "=" * 60)
    print("RESULTADOS")
    print("=" * 60)

    print(f"Total reviews: {total_reviews}")
    print(f"Total items únicos: {total_items}")

    print(f"\nItems sin ningún reviewText válido: {len(items_without_text)}")
    print(f"Items sin ningún summary válido: {len(items_without_summary)}")

    # Estadísticas de número de reviews por item
    review_counts = list(item_review_count.values())
    print("\nDistribución reviews por item:")
    print(f"  Min: {min(review_counts)}")
    print(f"  Max: {max(review_counts)}")
    print(f"  Media: {sum(review_counts) / len(review_counts):.2f}")

    # Items con pocas reviews
    few_reviews = sum(1 for c in review_counts if c <= 2)
    print(f"\nItems con ≤2 reviews: {few_reviews}")

    print("\nEjemplo de items sin texto:")
    print(items_without_text[:10])


if __name__ == "__main__":
    main()

In [ ]:
import pickle

# Dataset RL
with open("processed/dataset_d3rlpy.pkl", "rb") as f:
    obj = pickle.load(f)

train_arrays = obj["data"]["train"]
valid_arrays = obj["data"]["val"]   # ojo: tu split se llama "val", no "valid"

num_items = obj["num_items"]
obs_dim = train_arrays["observations"].shape[1]

# Embeddings de ítems
with open("processed/item_embeddings_hybrid.pkl", "rb") as f:
    emb_obj = pickle.load(f)

embedding_matrix = emb_obj["embedding_matrix"]

print("num_items:", num_items)
print("obs_dim:", obs_dim)
print("embedding_matrix shape:", embedding_matrix.shape)

In [ ]:
import numpy as np

data = np.load("processed/offline_rl_dataset.npz")

print("observations:", data["observations"].shape)
print("actions:", data["actions"].shape)
print("rewards mean:", data["rewards"].mean())
print("reward positive rate:", (data["rewards"] > 0).mean())
print("terminals mean:", data["terminals"].mean())
print("num_items:", int(data["num_items"]))

In [ ]:
# step10_validate_candidates.py
import numpy as np
import pandas as pd

df = pd.read_parquet("processed/events_with_split.parquet")
item_first_seen = df.groupby("item_id")["unixReviewTime"].min().sort_index().values

def check(split):
    cand = np.load(f"processed/{split}_candidates.npy")
    df_split = df[df["split"]==split].reset_index(drop=True)

    assert len(cand) == len(df_split)
    # positivo en col 0
    assert np.all(cand[:,0] == df_split["item_id"].values)

    t = df_split["unixReviewTime"].values
    # check first_seen <= t for all candidates
    first = item_first_seen[cand]
    ok = (first <= t[:,None]).all()
    print(split, "first_seen constraint:", ok)

    # no duplicates per row
    dup_rate = np.mean([len(set(row)) != len(row) for row in cand[:2000]])
    print(split, "dup_rate@2k:", dup_rate)

check("val")
check("test")

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))  # o la ruta raíz de tu proyecto

In [ ]:
import numpy as np
import torch
from src.eval import load_dataset_and_candidates, hr_ndcg_at_k_from_scores

val_obs, _, val_cand, _ = load_dataset_and_candidates()

# Tomamos solo 50k para probar
B = 50000
C = val_cand.shape[1]

# Scores completamente aleatorios
scores = torch.randn(B, C)

metrics = hr_ndcg_at_k_from_scores(scores, ks=[10])
print(metrics)

In [ ]:
import torch
import numpy as np
from src.eval import load_dataset_and_candidates, evaluate_split, make_score_fn_from_qnet_full
from src.train_bc import BCQNet  # o importa la clase correctamente

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = np.load("processed/dataset_rl_by_split.npz")
test_obs = data["test_observations"]
num_items = int(data["num_items"])

test_cand = np.load("processed/test_candidates.npy")

model = BCQNet(state_dim=128, num_items=num_items)
model.load_state_dict(torch.load("processed/bc_qnet.pt", map_location=device))
model.to(device).eval()

score_fn = make_score_fn_from_qnet_full(model, device=device)

metrics = evaluate_split(
    test_obs,
    test_cand,
    score_fn,
    ks=[5,10,20],
    batch_size=4096,
    device=str(device),
)

print("TEST:", metrics)